In [0]:
# ============================================================
# Top Sellers by Revenue
# ============================================================

display(
    spark.sql("""
        SELECT 
            s.seller_name,
            COUNT(DISTINCT f.invoice_number)        AS total_invoices,
            ROUND(SUM(f.item_total_price), 2)       AS total_line_item_revenue,
            ROUND(AVG(f.total), 2)                  AS avg_invoice_value,
            ROUND(SUM(f.tax), 2)                    AS total_tax_collected
        FROM invoices.gold.fact_invoices f
        JOIN invoices.gold.dim_seller s ON f.seller_key = s.seller_key
        GROUP BY s.seller_name
        ORDER BY total_line_item_revenue DESC
        LIMIT 10
    """)
)

In [0]:
# ============================================================
# Top Clients by Spend
# ============================================================

display(
    spark.sql("""
        SELECT 
            c.client_name,
            COUNT(DISTINCT f.invoice_number)   AS total_invoices,
            ROUND(SUM(f.total), 2)             AS total_spend,
            ROUND(AVG(f.total), 2)             AS avg_invoice_value,
            COUNT(f.item_description)          AS total_items_purchased
        FROM invoices.gold.fact_invoices f
        JOIN invoices.gold.dim_client c ON f.client_key = c.client_key
        GROUP BY c.client_name
        ORDER BY total_spend DESC
        LIMIT 10
    """)
)

In [0]:
# ============================================================
# Monthly Revenue Trend
# ============================================================

display(
    spark.sql("""
        SELECT 
            d.year,
            d.month,
            d.month_name,
            COUNT(DISTINCT f.invoice_number)    AS invoice_count,
            ROUND(SUM(f.total), 2)              AS monthly_revenue,
            ROUND(AVG(f.total), 2)              AS avg_invoice_value
        FROM invoices.gold.fact_invoices f
        JOIN invoices.gold.dim_date d ON f.date_key = d.date_key
        GROUP BY d.year, d.month, d.month_name
        ORDER BY d.year, d.month
    """)
)

In [0]:
# ============================================================
# Top 10 Best Selling Items by Revenue
# ============================================================

display(
    spark.sql("""
        SELECT 
            item_description,
            COUNT(*)                            AS times_sold,
            ROUND(SUM(item_total_price), 2)     AS total_revenue,
            ROUND(AVG(item_quantity), 2)        AS avg_quantity,
            ROUND(AVG(item_total_price), 2)     AS avg_item_price
        FROM invoices.gold.fact_invoices
        WHERE item_description IS NOT NULL
        GROUP BY item_description
        ORDER BY total_revenue DESC
        LIMIT 10
    """)
)

In [0]:
# ============================================================
# Invoice Aging (how old are unpaid invoices?)
# ============================================================

display(
    spark.sql("""
        SELECT 
            invoice_number,
            c.client_name,
            f.date_key                              AS invoice_date,
            f.invoice_due_date,
            DATEDIFF(current_date(), f.date_key)    AS days_since_invoice,
            ROUND(f.total, 2)                       AS invoice_total,
            CASE 
                WHEN DATEDIFF(current_date(), f.date_key) <= 30  THEN '0-30 days'
                WHEN DATEDIFF(current_date(), f.date_key) <= 60  THEN '31-60 days'
                WHEN DATEDIFF(current_date(), f.date_key) <= 90  THEN '61-90 days'
                ELSE 'Over 90 days'
            END AS aging_bucket
        FROM invoices.gold.fact_invoices f
        JOIN invoices.gold.dim_client c ON f.client_key = c.client_key
        WHERE f.invoice_due_date IS NULL          -- invoices with no due date set
           OR f.invoice_due_date < current_date() -- overdue invoices
        GROUP BY invoice_number, c.client_name, f.date_key, f.invoice_due_date, f.total
        ORDER BY days_since_invoice DESC
        LIMIT 20
    """)
)

In [0]:
# ============================================================
# Full Pipeline Health Summary
# ============================================================

display(
    spark.sql("""
        SELECT 'Bronze' AS layer, COUNT(*) AS row_count FROM invoices.bronze.raw_invoices
        UNION ALL
        SELECT 'Silver', COUNT(*) FROM invoices.silver.invoices_clean
        UNION ALL
        SELECT 'Gold - fact_invoices', COUNT(*) FROM invoices.gold.fact_invoices
        UNION ALL
        SELECT 'Gold - dim_client', COUNT(*) FROM invoices.gold.dim_client
        UNION ALL
        SELECT 'Gold - dim_seller', COUNT(*) FROM invoices.gold.dim_seller
        UNION ALL
        SELECT 'Gold - dim_date', COUNT(*) FROM invoices.gold.dim_date
    """)
)